# Fine-tune Qwen2.5-7B → MITRE ATT&CK Specialist (Colab)

**Runtime → Change runtime type → GPU (T4)** before running.

Flow: install Unsloth → clone repo → upload `train.jsonl`/`val.jsonl` → train QLoRA → export GGUF → download.
Then on your machine: `ollama create mitre-qwen:7b -f export/Modelfile`.

In [ ]:
# 1. Install Unsloth (CUDA-matched wheels for Colab)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Clone the repo (fine-tune branch) to get train_unsloth.py + ft_config.py
BRANCH = "feat/finetune-mitre-specialist"
REPO = "https://github.com/NitithX374/CyberCase-Intelligence-Framework.git"
!git clone --depth 1 -b $BRANCH $REPO
%cd CyberCase-Intelligence-Framework/rag_service/finetune

In [ ]:
# 3. Upload the dataset built locally (data/output is gitignored)
#    Run `python data/build_dataset.py --max-per-category 600` on your machine first.
import os
from google.colab import files
os.makedirs("data/output", exist_ok=True)
print("Select train.jsonl and val.jsonl ...")
up = files.upload()
for name in up:
    os.replace(name, f"data/output/{name}")
!wc -l data/output/*.jsonl

In [ ]:
# 4. Smoke test (5 steps) — make sure the loop runs before the full job
!python train/train_unsloth.py --max-steps 5

In [ ]:
# 5. Full training + GGUF export (Q4_K_M)
!python train/train_unsloth.py --gguf

In [ ]:
# 6. Download the GGUF to your machine
import glob
from google.colab import files
ggufs = glob.glob("export/outputs/gguf/*.gguf")
print("GGUF files:", ggufs)
for g in ggufs:
    if g.lower().endswith(("q4_k_m.gguf", "q4_k_m.gguf".upper())):
        files.download(g)

## On your machine (keeps the original model intact)
```powershell
cd rag_service/finetune
# put the downloaded .gguf in export/outputs/gguf/ and set FROM in export/Modelfile
ollama create mitre-qwen:7b -f export/Modelfile
ollama list                      # qwen2.5:7b AND mitre-qwen:7b
python compare/run_comparison.py --max-samples 20
```